# 🤖 Job Description Skills Extraction

MongoDB 및 덤프 파일(`data/silver_jobs.gz`)을 활용해 채용 정보 본문에서 핵심 기술 스택 및 키워드를 추출합니다.

### 1. 환경 설정 및 스파이시(SpaCy) 로드

In [ ]:
import os
import re
import pandas as pd
from pymongo import MongoClient
import spacy
from spacy.matcher import PhraseMatcher
from collections import Counter

# SpaCy 영어 모델 로드
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("⚠️ SpaCy 모델이 없습니다. 설치를 실행합니다...")
    import sys
    !{sys.executable} -m pip install spacy
    !{sys.executable} -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

### 2. 데이터 로드 (로컬 덤프 파일 확인 및 DB 연결)

In [ ]:
dump_path = "../data/silver_jobs.gz"
if os.path.exists(dump_path):
    print(f"✅ 로컬 덤프 파일 확인됨: {dump_path}")
    print("💡 덤프 복원 방법: 'mongorestore --gzip --archive=data/silver_jobs.gz' 실행 후 아래 셀을 구동하십시오.")

try:
    client = MongoClient("mongodb://mongodb:27017")
    db = client.linkedin
    jobs_coll = db['silver.jobs']
    df = pd.DataFrame(list(jobs_coll.find().limit(500))) # 500개 샘플링
    if not df.empty and '_id' in df.columns:
        df = df.drop(columns=['_id'])
    print(f"📊 분석 샘플 크기: {len(df)}개")
except Exception as e:
    print("❌ MongoDB 접속에 실패했습니다.", e)
    df = pd.DataFrame()

df.head(2)

### 3. NER 및 룰 기반 사전 정의 사전(PhraseMatcher) 정의

In [ ]:
tech_keywords = [
    "Python", "JavaScript", "TypeScript", "Java", "C++", "C#", "Go", "Rust", "Ruby", "PHP",
    "React", "Angular", "Vue", "Next.js", "NestJS", "Express", "Spring Boot", "Django", "FastAPI",
    "Docker", "Kubernetes", "AWS", "GCP", "Azure", "Git", "MongoDB", "MySQL", "PostgreSQL", "Redis",
    "Playwright", "Selenium", "Puppeteer", "HTML", "CSS", "Tailwind", "Terraform", "Ansible"
]

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(text) for text in tech_keywords]
matcher.add("TechStack", patterns)

def extract_skills(text):
    if not isinstance(text, str) or not text:
        return []
    
    clean_text = re.sub(r'---.*?---', '', text, flags=re.DOTALL)
    doc = nlp(clean_text)
    matches = matcher(doc)
    skills = set()
    for match_id, start, end in matches:
        span = doc[start:end]
        skills.add(span.text.capitalize())
    return list(skills)

### 4. 기술 스택 추출 적용

In [ ]:
if not df.empty:
    df['extracted_skills'] = df['description'].apply(extract_skills)
    display(df[['title', 'companyName', 'extracted_skills']].head(10))
else:
    print("데이터가 없습니다.")

### 5. 최다 매칭 스택 차트 분석

In [ ]:
if not df.empty:
    all_skills = []
    for skills in df['extracted_skills']:
        all_skills.extend(skills)
    
    skill_counts = Counter(all_skills).most_common(15)
    skill_df = pd.DataFrame(skill_counts, columns=['Skill', 'Count'])
    
    sns.barplot(data=skill_df, x='Count', y='Skill', palette='deep')
    plt.title("채용 공고 최다 요구 기술 스택 Top 15", fontsize=15)
    plt.xlabel("등장 횟수")
    plt.ylabel("기술 스택")
    plt.show()